In [5]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".keras"):
            print(os.path.join(root, file))

/kaggle/input/models/hemasakshis/dr-classification-model-/tensorflow2/default/1/final_dr_model.keras


In [7]:
import tensorflow as tf

MODEL_PATH = "/kaggle/input/models/hemasakshis/dr-classification-model-/tensorflow2/default/1/final_dr_model.keras"

cnn_model = tf.keras.models.load_model(MODEL_PATH)

print("CNN model loaded successfully!")
cnn_model.summary()

CNN model loaded successfully!


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 5)              │        10,245 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,574,545 (204.37 MB)

 Trainable params: 14,988,293 (57.18 MB)

 Non-trainable params: 8,609,664 (32.84 MB)

 Optimizer params: 29,976,588 (114.35 MB)

In [8]:
import os

DATASET_PATH = "/kaggle/input/competitions/aptos2019-blindness-detection"

print(os.listdir(DATASET_PATH))

['sample_submission.csv', 'train_images', 'train.csv', 'test.csv', 'test_images']


In [9]:
import pandas as pd
import os

# Load the dataset labels
csv_path = os.path.join(DATASET_PATH, "train.csv")
df = pd.read_csv(csv_path)

# Select the first retinal image
sample_image_name = df.iloc[0]["id_code"]
actual_label = df.iloc[0]["diagnosis"]

# Get its image path
image_path = os.path.join(
    DATASET_PATH,
    "train_images",
    sample_image_name + ".png"
)

print("Image name:", sample_image_name)
print("Actual DR grade:", actual_label)
print("Image path:", image_path)

Image name: 000c1434d8d7
Actual DR grade: 2
Image path: /kaggle/input/competitions/aptos2019-blindness-detection/train_images/000c1434d8d7.png


In [10]:
import tensorflow as tf

# Read the retinal image
image = tf.io.read_file(image_path)

# Decode the image into RGB format
image = tf.image.decode_png(image, channels=3)

# Resize to the model's expected input size
image = tf.image.resize(image, [224, 224])

# Apply ResNet50 preprocessing
image = tf.keras.applications.resnet50.preprocess_input(image)

# Add batch dimension
image = tf.expand_dims(image, axis=0)

print("Prepared image shape:", image.shape)
print("Image preprocessing completed!")

Prepared image shape: (1, 224, 224, 3)
Image preprocessing completed!


In [11]:
# Get predictions from the CNN
predictions = cnn_model.predict(image)

# Get the class with the highest probability
predicted_class = predictions.argmax(axis=1)[0]

# Get the confidence score
confidence = predictions[0][predicted_class]

# DR class names
class_names = [
    "No DR",
    "Mild",
    "Moderate",
    "Severe",
    "Proliferative DR"
]

print("Predicted DR Grade:", class_names[predicted_class])
print("Confidence:", round(float(confidence) * 100, 2), "%")

print("\nClass-wise probabilities:")

for i, probability in enumerate(predictions[0]):
    print(class_names[i], ":", round(float(probability) * 100, 2), "%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step
Predicted DR Grade: Moderate
Confidence: 69.47 %

Class-wise probabilities:
No DR : 0.02 %
Mild : 2.55 %
Moderate : 69.47 %
Severe : 4.39 %
Proliferative DR : 23.56 %


In [12]:
print("Actual DR Grade:", class_names[int(actual_label)])
print("Predicted DR Grade:", class_names[predicted_class])

if int(actual_label) == predicted_class:
    print("Result: Correct Prediction")
else:
    print("Result: Incorrect Prediction")

Actual DR Grade: Moderate
Predicted DR Grade: Moderate
Result: Correct Prediction


In [13]:
# Convert CNN prediction into structured data
cnn_output = {
    "predicted_class": class_names[predicted_class],
    "confidence": float(confidence),
    "probabilities": {
        class_names[i]: float(predictions[0][i])
        for i in range(len(class_names))
    }
}

print(cnn_output)

{'predicted_class': 'Moderate', 'confidence': 0.6947429180145264, 'probabilities': {'No DR': 0.00021847766765858978, 'Mild': 0.025516953319311142, 'Moderate': 0.6947429180145264, 'Severe': 0.04388175532221794, 'Proliferative DR': 0.23563992977142334}}


In [26]:
!pip install groq

In [28]:
from groq import Groq
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
groq_api_key = secrets.get_secret("DR_G_Report")

groq_client = Groq(api_key=groq_api_key)

print("Groq connected successfully!")

Groq connected successfully!


In [29]:
MODEL_NAME = "openai/gpt-oss-120b"

test_response = groq_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": "Write one sentence explaining diabetic retinopathy."
        }
    ],
    temperature=0.2
)

print(test_response.choices[0].message.content)

Diabetic retinopathy is a diabetes‑related eye disease in which high blood‑sugar damage to the retinal blood vessels causes leakage, swelling, and abnormal growth of new vessels, leading to progressive vision loss if untreated.


In [36]:
import numpy as np

# Get probabilities for the sample image
probs = predictions[0]

# Sort classes by probability
sorted_indices = np.argsort(probs)[::-1]

# Top prediction
top_index = sorted_indices[0]
top_class = class_names[top_index]
top_probability = float(probs[top_index])

# Second-highest prediction
second_index = sorted_indices[1]
second_class = class_names[second_index]
second_probability = float(probs[second_index])

# Probability difference
top_two_gap = top_probability - second_probability

# Normalized prediction entropy
entropy = -np.sum(probs * np.log(probs + 1e-10)) / np.log(5)

# Create structured CNN output
cnn_output = {
    "predicted_grade": int(top_index),
    "predicted_class": top_class,
    "softmax_probabilities": {
        class_names[i]: float(probs[i])
        for i in range(5)
    },
    "maximum_probability": top_probability,
    "top_two_classes": [
        {
            "class": top_class,
            "probability": top_probability
        },
        {
            "class": second_class,
            "probability": second_probability
        }
    ],
    "top_two_probability_gap": float(top_two_gap),
    "normalized_entropy": float(entropy)
}

print(cnn_output)

{'predicted_grade': 2, 'predicted_class': 'Moderate', 'softmax_probabilities': {'No DR': 0.00021847766765858978, 'Mild': 0.025516953319311142, 'Moderate': 0.6947429180145264, 'Severe': 0.04388175532221794, 'Proliferative DR': 0.23563992977142334}, 'maximum_probability': 0.6947429180145264, 'top_two_classes': [{'class': 'Moderate', 'probability': 0.6947429180145264}, {'class': 'Proliferative DR', 'probability': 0.23563992977142334}], 'top_two_probability_gap': 0.459102988243103, 'normalized_entropy': 0.5133930730698856}


In [37]:
print("cnn_output available:", "cnn_output" in globals())
print("predictions available:", "predictions" in globals())
print("class_names available:", "class_names" in globals())

cnn_output available: True
predictions available: True
class_names available: True


In [38]:
import json

common_cnn_input = json.dumps(cnn_output, indent=4)

print(common_cnn_input)

{
    "predicted_grade": 2,
    "predicted_class": "Moderate",
    "softmax_probabilities": {
        "No DR": 0.00021847766765858978,
        "Mild": 0.025516953319311142,
        "Moderate": 0.6947429180145264,
        "Severe": 0.04388175532221794,
        "Proliferative DR": 0.23563992977142334
    },
    "maximum_probability": 0.6947429180145264,
    "top_two_classes": [
        {
            "class": "Moderate",
            "probability": 0.6947429180145264
        },
        {
            "class": "Proliferative DR",
            "probability": 0.23563992977142334
        }
    ],
    "top_two_probability_gap": 0.459102988243103,
    "normalized_entropy": 0.5133930730698856
}


In [41]:
zero_shot_prompt = f"""
You are an AI-assisted medical report writing assistant
for an experimental Diabetic Retinopathy screening system.

Your task is to generate a clear, professional and
scientifically cautious screening report using ONLY
the CNN output provided below.

CNN OUTPUT:
{common_cnn_input}

Follow this report structure:

1. Screening Result
- State the predicted DR grade and category.
- Clearly identify it as a CNN-generated prediction.

2. Class-wise Probability Distribution
- Include all five DR categories.
- Convert probabilities into percentages.
- Identify the highest and second-highest predicted classes.
- Ensure the values accurately match the CNN output.

3. Prediction Uncertainty
- Explain the maximum softmax probability.
- Explain the difference between the top two class probabilities.
- Explain normalized entropy as an experimental
  indicator of prediction uncertainty.
- Do not describe these values as calibrated clinical
  confidence or actual disease probabilities.
- Do not make unsupported claims about disease severity.

4. Model Limitations
- State that this is an experimental AI model.
- Explain that it has not been clinically validated.
- Clarify that the LLM only interprets numerical CNN
  outputs and does not examine the retinal image.
- Do not invent retinal findings, patient information,
  symptoms, medical history or clinical observations.

5. Recommendation
- Recommend assessment of the original retinal image
  by a qualified ophthalmologist or eye-care professional.
- Clarify that the prediction is not a confirmed diagnosis.

Additional requirements:
- Use professional, simple medical language.
- Avoid unnecessary repetition.
- Do not recommend medication or treatment.
- Do not claim that the AI can replace a clinician.
- Do not add information that is absent from the CNN output.

Generate a complete, well-organized screening report.
"""

In [42]:
response_zero_shot = groq_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": zero_shot_prompt
        }
    ],
    temperature=0.2
)

report_zero_shot = response_zero_shot.choices[0].message.content

print(report_zero_shot)

**Diabetic Retinopathy Screening Report**  
*Generated from experimental CNN output (no image review performed)*  

---

### 1. Screening Result
- **Predicted DR grade:** **2 (Moderate)**  
- This prediction is produced by a convolutional neural network (CNN) model and reflects the model’s most likely classification for the supplied retinal image.

---

### 2. Class‑wise Probability Distribution
| DR Category | Softmax Probability | Percentage |
|-------------|--------------------|------------|
| No DR | 0.000218 | **0.02 %** |
| Mild | 0.025517 | **2.55 %** |
| **Moderate** | **0.694743** | **69.47 %** |
| Severe | 0.043882 | **4.39 %** |
| Proliferative DR | 0.235640 | **23.56 %** |

- **Highest‑probability class:** *Moderate* (≈ 69.5 %).  
- **Second‑highest‑probability class:** *Proliferative DR* (≈ 23.6 %).  

---

### 3. Prediction Uncertainty
- **Maximum softmax probability:** The model assigns a probability of **0.6947** (≈ 69.5 %) to the *Moderate* class. This is the strongest

In [43]:
with open("report_zero_shot.txt", "w", encoding="utf-8") as f:
    f.write(report_zero_shot)

print("Improved Zero-Shot report saved successfully!")

Improved Zero-Shot report saved successfully!


In [46]:

few_shot_prompt = f"""
You are an AI-assisted medical report writing assistant
for an experimental Diabetic Retinopathy screening system.

Your task is to generate a clear, professional and
scientifically cautious screening report using ONLY
the CNN prediction information provided below.

Follow the style, structure and level of detail demonstrated
in the examples.

EXAMPLE 1:

CNN OUTPUT:
{{
    "predicted_class": "No DR",
    "softmax_probabilities": {{
        "No DR": 0.92,
        "Mild": 0.04,
        "Moderate": 0.02,
        "Severe": 0.01,
        "Proliferative DR": 0.01
    }},
    "maximum_probability": 0.92
}}

SAMPLE REPORT:

Predicted Result: No DR.

The CNN assigns the highest softmax probability to the
No DR category (92%). The remaining probabilities are
distributed across the other four categories.

This is an experimental model prediction and does not
confirm the absence of diabetic retinopathy. Review by
a qualified eye-care professional is recommended.


EXAMPLE 2:

CNN OUTPUT:
{{
    "predicted_class": "Moderate",
    "softmax_probabilities": {{
        "No DR": 0.01,
        "Mild": 0.10,
        "Moderate": 0.70,
        "Severe": 0.12,
        "Proliferative DR": 0.07
    }},
    "maximum_probability": 0.70
}}

SAMPLE REPORT:

Predicted Result: Moderate DR.

The CNN assigns the highest softmax probability to
the Moderate category (70%). Mild and Severe are the
next most probable categories.

This output represents an experimental model prediction,
not a confirmed diagnosis. Assessment by a qualified
eye-care professional is recommended.


NOW GENERATE A REPORT FOR THE FOLLOWING CNN OUTPUT:

CNN OUTPUT:
{common_cnn_input}


REPORT STRUCTURE AND INSTRUCTIONS:

1. Screening Result
- State the predicted DR grade and category.
- Identify it as a CNN-generated prediction.

2. Class-wise Probability Distribution
- Include all five DR categories.
- Convert probabilities into percentages.
- Identify the highest and second-highest predicted classes.
- Ensure all values accurately match the CNN output.

3. Prediction Uncertainty
- Explain the maximum softmax probability.
- Explain the difference between the top two class probabilities
  in percentage points.
- Explain normalized entropy as an experimental indicator
  describing the distribution of predicted probabilities.
- Do not interpret entropy as calibrated confidence or
  clinical certainty.
- Do not make unsupported claims about disease severity.

4. Model Limitations
- State that this is an experimental research-stage model
  that has not undergone clinical validation.
- Clarify that the CNN processed the retinal image to
  generate the prediction.
- Clarify that the language model received only the
  numerical CNN output and did not independently examine
  the retinal image.
- Do not invent retinal findings, patient details,
  symptoms or medical history.

5. Recommendation
- Recommend review of the original retinal image and
  clinical information by a qualified ophthalmologist
  or retinal specialist.
- Clarify that this prediction is not a confirmed diagnosis.

ADDITIONAL REQUIREMENTS:
- Follow the professional style demonstrated in the examples.
- Use simple, clear medical language.
- Avoid unnecessary repetition.
- Do not recommend medication or treatment.
- Do not claim that AI can replace a clinician.
- Do not add information that is absent from the CNN output.

Generate a complete, well-organized screening report.
"""

print("Improved Few-Shot prompt created successfully!")


Improved Few-Shot prompt created successfully!


In [47]:
response_few_shot = groq_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": few_shot_prompt
        }
    ],
    temperature=0.2
)

report_few_shot = response_few_shot.choices[0].message.content

print(report_few_shot)

**Screening Result**  
The CNN predicts a **moderate diabetic retinopathy (DR) grade 2** for the examined retinal image. This is a model‑generated prediction and not a definitive clinical diagnosis.  

---

**Class‑wise Probability Distribution**  

| DR Category            | Softmax Probability | Percentage |
|------------------------|---------------------|------------|
| No DR                  | 0.0002185           | **0.02 %** |
| Mild                   | 0.0255170           | **2.55 %** |
| **Moderate** (predicted) | **0.6947429**       | **69.47 %** |
| Severe                 | 0.0438818           | **4.39 %** |
| Proliferative DR       | 0.2356399           | **23.56 %** |

- **Highest probability:** Moderate (69.47 %).  
- **Second‑highest probability:** Proliferative DR (23.56 %).  

---

**Prediction Uncertainty**  

- The **maximum softmax probability** is 0.6947 (≈ 69.5 %), indicating the model’s strongest assignment to the Moderate class.  
- The **gap between the top two c

In [48]:

report_few_shot = report_few_shot.replace(
    "a value around 0.5 suggests a moderate level of uncertainty.",
    "the probability distribution is spread across multiple "
    "classes, although the Moderate category receives the "
    "largest share. No clinically validated uncertainty "
    "threshold has been established for this metric."
)

with open("report_few_shot.txt", "w", encoding="utf-8") as f:
    f.write(report_few_shot)

print("Experiment 2 - Few-Shot report saved successfully!")


Experiment 2 - Few-Shot report saved successfully!


In [51]:
# this is uncertainitiy aware adaption prompt
uncertainty_aware_prompt = f"""
You are an AI-assisted medical report writing assistant
for an experimental Diabetic Retinopathy screening system.

Your task is to generate a professional, scientifically
cautious screening report using ONLY the numerical output
provided by the CNN.

CNN OUTPUT:
{common_cnn_input}


CORE APPROACH: UNCERTAINTY-AWARE ADAPTIVE REPORTING

Adapt the explanation of the CNN prediction by considering
the following three indicators together.

1. MAXIMUM SOFTMAX PROBABILITY

- Identify the highest softmax probability and its class.
- Explain that it represents the largest score assigned
  by the CNN among the five classes.
- Do not describe it as calibrated clinical confidence
  or a confirmed probability of disease.

2. TOP-TWO PROBABILITY GAP

- Identify the two highest probability classes.
- Explain the numerical difference between their scores.
- Express the difference in percentage points.
- Explain whether the leading class is numerically
  separated from or relatively close to the second class.
- Do not interpret this gap as clinical certainty.

3. NORMALIZED ENTROPY

- Use the supplied normalized entropy value.
- Explain that entropy describes how the CNN's
  probability distribution is spread across the classes.
- A more concentrated distribution has lower entropy,
  while a more evenly distributed probability pattern
  has higher entropy.
- Do not assign labels such as low, moderate or high
  uncertainty using unvalidated thresholds.
- Do not interpret entropy as clinical certainty.

4. COMBINED ADAPTIVE INTERPRETATION

- Consider maximum softmax probability, top-two gap
  and normalized entropy together.
- Explain which class receives the largest probability.
- Describe the separation between the two leading classes.
- Explain how the remaining probabilities contribute
  to the overall distribution.
- Avoid contradictory or exaggerated interpretations.
- Do not invent additional uncertainty measurements.


REPORT STRUCTURE:

1. SCREENING RESULT

- State the predicted DR grade and category.
- Clearly identify it as a CNN-generated prediction.
- Do not present it as a confirmed diagnosis.

2. CLASS-WISE PROBABILITY DISTRIBUTION

- Include all five DR categories.
- Convert probabilities into percentages.
- Round percentages to two decimal places.
- Identify the highest and second-highest classes.
- Preserve the numerical values supplied by the CNN.

3. ADAPTIVE UNCERTAINTY ANALYSIS

Include four subsections:

a. Maximum Softmax Probability
b. Top-Two Probability Gap
c. Normalized Entropy
d. Combined Interpretation

Explain all three indicators accurately and describe
their relationship without making clinical conclusions.

4. MODEL LIMITATIONS

- State that the CNN is an experimental research-stage
  model that has not undergone formal clinical validation.
- Clarify that the CNN processed the retinal image.
- Clarify that the language model received only the
  numerical CNN output and did not independently
  examine the retinal image.
- Explain that the generated report does not include
  direct visual observations from the language model.
- Do not invent patient information, symptoms,
  medical history or retinal findings.

5. RECOMMENDATION

- Recommend review of the original retinal image and
  relevant clinical information by a qualified
  ophthalmologist or retinal specialist.
- Clearly state that the CNN prediction is not a
  confirmed diagnosis.
- Do not recommend medication or treatment.


STRICT ACCURACY AND SAFETY REQUIREMENTS:

- Use only the supplied CNN output.
- Do not change, omit or fabricate probability values.
- Do not invent retinal lesions or clinical observations.
- Do not describe softmax scores as calibrated
  clinical probabilities.
- Do not apply unvalidated clinical thresholds.
- Do not claim that AI can replace an ophthalmologist.
- Use clear, professional and scientifically cautious
  language.
- Avoid unnecessary repetition.


Generate a complete, well-organized screening report
with an adaptive interpretation of the CNN's numerical
prediction and uncertainty indicators.
"""

print("Experiment 3 - Corrected Uncertainty-Aware prompt created!")


Experiment 3 - Corrected Uncertainty-Aware prompt created!


In [52]:

response_uncertainty = groq_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": uncertainty_aware_prompt
        }
    ],
    temperature=0.2
)

report_uncertainty = response_uncertainty.choices[0].message.content

print(report_uncertainty)


**Diabetic Retinopathy Screening Report – Adaptive AI Interpretation**  

---

### 1. SCREENING RESULT  
- **Predicted DR grade:** **2** (moderate)  
- **Predicted category (CNN output):** **“Moderate”**  

*The above grade is the class with the highest soft‑max score generated by the experimental convolutional neural network (CNN). It reflects the model’s internal ranking of the five possible categories and is **not** a confirmed clinical diagnosis.*

---

### 2. CLASS‑WISE PROBABILITY DISTRIBUTION  

| DR Category            | Soft‑max probability | Percentage* |
|------------------------|----------------------|-------------|
| No DR                  | 0.0002185            | **0.02 %** |
| Mild                   | 0.0255170            | **2.55 %** |
| **Moderate** (highest) | **0.6947429**        | **69.47 %** |
| Severe                 | 0.0438818            | **4.39 %** |
| **Proliferative DR** (second highest) | **0.2356399** | **23.56 %** |

\*Percentages are the soft‑max probabi

In [53]:

with open("report_uncertainty_aware.txt", "w", encoding="utf-8") as f:
    f.write(report_uncertainty)

print("Experiment 3 report saved successfully!")


Experiment 3 report saved successfully!


In [54]:

cov_prompt = f"""
You are an AI-assisted medical report writing assistant
for an experimental Diabetic Retinopathy screening system.

Your task is to generate an accurate and scientifically
cautious screening report using ONLY the numerical output
provided by the CNN.

ORIGINAL CNN OUTPUT:
{common_cnn_input}


CHAIN-OF-VERIFICATION PROCESS

Follow these four stages carefully.

STAGE 1: INITIAL REPORT GENERATION

Generate an initial screening report using the CNN output.

Include:
- Predicted DR grade and category.
- All five class-wise probabilities.
- Maximum softmax probability.
- Top-two predicted classes and their probability gap.
- Normalized entropy.
- Model limitations and recommendation.

STAGE 2: CREATE VERIFICATION QUESTIONS

Identify factual claims in your initial report that need
verification.

Create verification questions covering:

1. Is the predicted DR grade consistent with the CNN output?
2. Are all five class probabilities copied correctly?
3. Are the highest and second-highest classes correct?
4. Is the maximum softmax probability accurate?
5. Is the top-two probability gap calculated correctly?
6. Is the normalized entropy copied correctly?
7. Does the report contain any unsupported clinical claims?
8. Does the report incorrectly describe the LLM as having
   examined the retinal image?

STAGE 3: VERIFY THE CLAIMS

Answer each verification question by comparing the
initial report directly against the ORIGINAL CNN OUTPUT.

Rules:
- Treat the original CNN output as the source of truth.
- Correct any incorrect numerical values.
- Do not invent missing information.
- Remove claims that cannot be supported by the CNN output.
- Do not interpret softmax scores as calibrated clinical
  probabilities or confirmed disease likelihood.
- Do not assign clinical uncertainty categories using
  unvalidated thresholds.
- Do not invent retinal lesions, symptoms, patient history,
  or other clinical observations.

STAGE 4: GENERATE THE VERIFIED FINAL REPORT

Use the verification results to produce a corrected report.

The final report must contain:

1. SCREENING RESULT
   - Predicted DR category and grade.
   - Clearly identify it as a CNN-generated prediction,
     not a confirmed diagnosis.

2. CLASS-WISE PROBABILITY DISTRIBUTION
   - Include all five classes.
   - Express probabilities as percentages rounded to
     two decimal places.
   - Preserve the CNN values accurately.

3. PREDICTION AND UNCERTAINTY INFORMATION
   - Maximum softmax probability.
   - Top-two predicted classes.
   - Top-two probability gap in percentage points.
   - Normalized entropy.
   - Explain these indicators without claiming clinical
     certainty or applying unvalidated thresholds.

4. MODEL LIMITATIONS
   - The CNN processed the retinal image.
   - The language model received only numerical CNN output
     and did not independently examine the image.
   - The system is experimental and has not undergone
     formal clinical validation.
   - The report does not establish a diagnosis.

5. RECOMMENDATION
   - Recommend review of the original retinal image and
     relevant clinical information by a qualified
     ophthalmologist or retinal specialist.
   - Do not recommend medication or treatment.

STRICT REQUIREMENTS:

- The original CNN output is the only source of prediction
  data.
- Do not modify or fabricate numerical values.
- Do not invent clinical findings or patient information.
- Do not claim that the LLM independently analyzed the
  retinal image.
- Do not claim that verification proves the CNN prediction
  is medically correct.
- Use professional, clear and cautious language.

OUTPUT FORMAT:

A. Initial Report
B. Verification Questions and Answers
C. Corrections Made
D. Final Verified Screening Report

Show all four sections so that the verification process
can be examined as part of this research experiment.
"""

print("Experiment 4 - Chain-of-Verification prompt created!")


Experiment 4 - Chain-of-Verification prompt created!


In [55]:

cov_response = groq_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": cov_prompt
        }
    ],
    temperature=0.2
)

report_cov = cov_response.choices[0].message.content

print(report_cov)


**A. Initial Report**

**Screening Summary**  
- **Predicted DR grade:** 2  
- **Predicted DR category:** Moderate  

**Class‑wise Softmax Probabilities**  
- No DR: 0.00021847766765858978  
- Mild: 0.025516953319311142  
- Moderate: 0.6947429180145264  
- Severe: 0.04388175532221794  
- Proliferative DR: 0.23563992977142334  

**Prediction Confidence Metrics**  
- **Maximum softmax probability:** 0.6947429180145264  
- **Top‑two predicted classes:**  
  1. Moderate (0.6947429180145264)  
  2. Proliferative DR (0.23563992977142334)  
- **Probability gap between top two classes:** 0.459102988243103 (≈45.91 percentage points)  
- **Normalized entropy:** 0.5133930730698856  

**Model Limitations & Recommendation**  
- The CNN processed the retinal image and generated the numerical predictions shown above.  
- The language model (this assistant) received only these numbers; it did **not** examine the image itself.  
- This system is experimental, has not undergone formal clinical validatio

In [56]:

with open("report_chain_of_verification.txt", "w", encoding="utf-8") as f:
    f.write(report_cov)

print("Experiment 4 - Chain-of-Verification report saved successfully!")


Experiment 4 - Chain-of-Verification report saved successfully!


In [57]:

evidence_grounded_prompt = f"""
You are an AI-assisted medical report writing assistant
for an experimental Diabetic Retinopathy screening system.

Your task is to generate a professional screening report
using a STRICT EVIDENCE-GROUNDED, CLOSED-WORLD approach.

ORIGINAL CNN OUTPUT:
{common_cnn_input}


CORE PRINCIPLE: CLOSED-WORLD INFORMATION

The supplied CNN output is the ONLY source of information
about this individual screening prediction.

You must follow these rules:

1. Use only information explicitly available in the
   supplied CNN output.

2. Do not use assumptions, external medical knowledge,
   or invented details to fill missing patient information.

3. Do not invent:
   - Patient name, age or gender.
   - Symptoms or medical history.
   - Retinal lesions or image observations.
   - Blood sugar levels or other test results.
   - Previous diagnoses or treatments.

4. The CNN processed the retinal image. You received only
   its numerical output and did not independently examine
   the retinal image.

5. If a requested detail is not available in the CNN output,
   clearly state:
   "Not available in the supplied CNN output."

6. Distinguish between a model prediction and a confirmed
   medical diagnosis.


REPORT REQUIREMENTS

1. SCREENING RESULT

- State the predicted DR grade and category.
- Use the exact predicted class from the CNN output.
- Clearly identify it as a CNN-generated prediction.
- Do not call it a confirmed diagnosis.

2. CLASS-WISE PROBABILITY DISTRIBUTION

- Include all five DR categories:
  No DR, Mild, Moderate, Severe and Proliferative DR.
- Convert the supplied probabilities into percentages.
- Round percentages to two decimal places.
- Preserve the original values accurately.

3. MODEL OUTPUT INTERPRETATION

- State the maximum softmax probability.
- Identify the top two predicted classes.
- State the top-two probability gap.
- State normalized entropy.
- Explain that these are numerical characteristics
  of the CNN output, not calibrated clinical probabilities.
- Do not apply unvalidated uncertainty thresholds.

4. AVAILABLE AND UNAVAILABLE INFORMATION

Clearly distinguish:

Available:
- CNN-predicted DR category and grade.
- Class-wise softmax scores.
- Maximum softmax probability.
- Top-two probability gap.
- Normalized entropy.

Unavailable:
- Patient identity and demographics.
- Symptoms and medical history.
- Direct clinical observations.
- Retinal lesion descriptions.
- Confirmatory examination results.

Do not infer any unavailable information.

5. MODEL LIMITATIONS

- This is an experimental research-stage system.
- It has not undergone formal clinical validation.
- The language model received only numerical CNN output.
- The language model did not independently examine
  the retinal image.
- This report is not a confirmed diagnosis.

6. RECOMMENDATION

- Recommend that a qualified ophthalmologist or retinal
  specialist review the original retinal image and
  relevant clinical information.
- Do not recommend medication or treatment.


FINAL ACCURACY CHECK

Before producing the report, check that:

- Every prediction-related statement is supported by
  the original CNN output.
- All five probabilities are accurately represented.
- No patient information has been invented.
- No retinal findings have been fabricated.
- No unsupported clinical conclusions are included.
- Missing information is clearly identified.
- The CNN prediction is not presented as a diagnosis.

OUTPUT FORMAT

A. Screening Result
B. Class-Wise Probability Distribution
C. Model Output Interpretation
D. Available and Unavailable Information
E. Model Limitations
F. Recommendation

Generate a complete, professional and scientifically
cautious screening report.
"""

print("Experiment 5 - Evidence-Grounded prompt created!")


Experiment 5 - Evidence-Grounded prompt created!


In [58]:

evidence_response = groq_client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {
            "role": "user",
            "content": evidence_grounded_prompt
        }
    ],
    temperature=0.2
)

report_evidence = evidence_response.choices[0].message.content

print(report_evidence)


**A. Screening Result**  
- **CNN‑predicted diabetic retinopathy (DR) grade:** 2  
- **CNN‑predicted DR category:** *Moderate*  
- This classification is generated by the convolutional neural network (CNN) model and **does not constitute a confirmed clinical diagnosis**.

---

**B. Class‑Wise Probability Distribution**  

| DR Category          | Softmax Probability (raw) | Percentage* |
|----------------------|---------------------------|-------------|
| No DR                | 0.00021847766765858978    | 0.02 % |
| Mild                 | 0.025516953319311142      | 2.55 % |
| Moderate             | 0.6947429180145264        | 69.47 % |
| Severe               | 0.04388175532221794       | 4.39 % |
| Proliferative DR     | 0.23563992977142334       | 23.56 % |

\*Percentages are the raw softmax values multiplied by 100 and rounded to two decimal places.

---

**C. Model Output Interpretation**  

- **Maximum softmax probability:** 0.6947429180145264 (corresponding to the *Moderate* clas

In [59]:

with open("report_evidence_grounded.txt", "w", encoding="utf-8") as f:
    f.write(report_evidence)

print("Experiment 5 - Evidence-Grounded report saved successfully!")


Experiment 5 - Evidence-Grounded report saved successfully!
